# Empirical Methods for Lawyers — Hands-on Statistics
# שיטות מחקר אמפיריות למשפטנים — שיעור מעשי בסטטיסטיקה

<div dir="rtl" lang="he">

<p>במחברת הזו נעבוד עם נתונים על <b>כ-16,000 תיקים של בית המשפט העליון</b> שנפתחו בשנים 2010–2017.
המטרה היא לא ללמוד לתכנת, אלא ללמוד <b>לחשוב עם נתונים</b>: מה בדיוק נמדד, מה חסר ולמה, כמה אפשר לסמוך על מספר, ומתי קשר בין שני משתנים אומר משהו.</p>
<p><b>מה נעשה:</b></p>
<ol>
<li>נכיר את הנתונים ואת סוגי המשתנים</li>
<li>נבין נתונים חסרים ושגיאות</li>
<li>נתאר משתנים: ממוצע, חציון, פיזור והתפלגות</li>
<li>נאמוד אי-ודאות: רווח סמך</li>
<li>נבחן קשרים בין משתנים ומתאמים</li>
<li>נערוך מבחני השערות</li>
<li>נשאל שאלות משלכם</li>
</ol>
<p><b>מקור הנתונים:</b> מאגר פסקי הדין של בית המשפט העליון (Israeli Supreme Court Database): כל ההחלטות הסופיות שניתנו בהרכב של שלושה שופטים ומעלה. Weinshall, Epstein &amp; Worms, <i>The Israeli Supreme Court Database</i>, 2018 version, <a href="https://web.archive.org/web/20221206211948/https://iscd.huji.ac.il/documentation">מילון המשתנים המלא</a>.</p>
<p><b>איך עובדים:</b> בתפריט בחרו <b>Runtime → Run all</b>. בתאים שיש בהם טופס (תפריטים בצד), שנו את הבחירה ולחצו ▶ כדי להריץ מחדש. לא צריך לכתוב קוד, אבל מותר ואפילו רצוי לנסות.</p>

</div>

In this notebook we work with data on **about 16,000 Israeli Supreme Court cases** filed between 2010 and 2017.
The goal is not to learn programming but to learn to **think with data**: what exactly was measured, what is missing and why,
how much we can trust a number, and when a relationship between two variables means something.

**Data source:** the Israeli Supreme Court Database — all final decisions given by panels of three or more justices.
Weinshall, Epstein & Worms, *The Israeli Supreme Court Database*, 2018 version ([full codebook](https://web.archive.org/web/20221206211948/https://iscd.huji.ac.il/documentation)).

**How to work:** choose **Runtime → Run all**. In cells with a form (menus on the side), change the selection and press ▶ to re-run.
You don't need to write code — but you are welcome to try.

## 0. Setup · הכנות

In [ ]:
#@title 0.1 Setup — run this first | הגדרות — להריץ ראשון
import os, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display, HTML

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

# Chart style: one main color, one comparison color, quiet axes
BLUE, ORANGE, GREY = "#2a78d6", "#eb6834", "#8a8a85"
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "figure.figsize": (8, 4.5), "axes.spines.top": False, "axes.spines.right": False,
    "axes.titleweight": "bold", "axes.titlesize": 12, "grid.alpha": 0.4,
})

# ---------- Small helpers used throughout (you don't need to read these) ----------
def say(en, he=None):
    """Print an explanation in English and (optionally) Hebrew."""
    html = f"<p style='margin:2px 0'>{en}</p>"
    if he:
        html += f"<p dir='rtl' lang='he' style='margin:2px 0'>{he}</p>"
    display(HTML(html))

def check_cols(*cols):
    for c in cols:
        if c not in df.columns:
            raise KeyError(f"'{c}' is not a column. Run `df.columns` to see the names. | העמודה '{c}' לא קיימת.")

def subset(data, merits_only=False, one_row_per_decision=False, filter_col="(none)", filter_val=""):
    """Choose which cases enter the analysis."""
    d = data
    if merits_only:
        d = d[d["dispositioncourt"] == "On the merits"]
    if filter_col and filter_col != "(none)":
        check_cols(filter_col)
        d = d[d[filter_col].astype(str) == str(filter_val)]
    if one_row_per_decision:
        d = d.drop_duplicates("url")
    return d

def mean_ci(x, conf=0.95):
    x = pd.Series(x).dropna()
    n = len(x)
    m = x.mean()
    se = x.std(ddof=1) / np.sqrt(n)
    t = stats.t.ppf((1 + conf) / 2, n - 1)
    return m, m - t * se, m + t * se, se, n

def is_continuous(s, max_levels=10):
    """Treat a column as numeric only if it is numeric AND has many distinct values."""
    return pd.api.types.is_numeric_dtype(s) and s.nunique() > max_levels

def cramers_v(ct):
    chi2 = stats.chi2_contingency(ct)[0]
    n = ct.to_numpy().sum()
    k = min(ct.shape) - 1
    return np.sqrt(chi2 / (n * k)) if k > 0 else np.nan

def p_sentence(p, alpha=0.05):
    if p < alpha:
        say(f"<b>p = {p:.4f} &lt; {alpha}</b>: if there were truly no difference in the population, a result this extreme would be rare. We <b>reject</b> H0.",
            f"<b>p = {p:.4f} &lt; {alpha}</b>: אילו באמת לא היה הבדל באוכלוסייה, תוצאה קיצונית כזו הייתה נדירה. אנו <b>דוחים</b> את השערת האפס.")
    else:
        say(f"<b>p = {p:.4f} ≥ {alpha}</b>: a result like this could easily happen by chance. We <b>do not reject</b> H0 (this is not proof that there is no difference).",
            f"<b>p = {p:.4f} ≥ {alpha}</b>: תוצאה כזו יכולה בקלות לנבוע ממקריות. איננו דוחים את השערת האפס (וזו אינה הוכחה שאין הבדל).")

print("Ready ✓  | מוכן ✓")

## 1. Meet the data · הכירו את הנתונים

<div dir="rtl" lang="he">

<p>לפני כל חישוב, שלוש שאלות: <b>מהי יחידת התצפית</b> (מה מייצגת שורה אחת)? <b>אילו משתנים יש</b> ומה כל אחד מודד? <b>על איזו אוכלוסייה</b> הנתונים מלמדים?</p>

</div>

Before computing anything, ask three questions: **What is the unit of observation** (what does one row represent)?
**Which variables are there** and what does each one measure? **What population** do the data tell us about?

In [ ]:
#@title 1.1 Load the data | טעינת הנתונים
#@markdown The link to the course data is already filled in. To use another file, paste a link (GitHub or a public Google Drive link), or leave it empty to upload a file from your computer.
#@markdown <br>הקישור לנתוני הקורס כבר מוזן. כדי להשתמש בקובץ אחר, הדביקו קישור או השאירו ריק כדי להעלות קובץ מהמחשב.
DATA_URL = "https://raw.githubusercontent.com/amit-haim/empirical-methods-law/main/data/data.csv" #@param {type:"string"}

def direct_link(url):
    """Turn a Google Drive 'share' link or a GitHub page link into a direct download link."""
    m = re.search(r"drive\.google\.com/(?:file/d/|open\?id=|uc\?.*id=)([\w-]+)", url)
    if m:
        return f"https://drive.google.com/uc?export=download&id={m.group(1)}"
    if "github.com" in url and "/blob/" in url:
        return url.replace("github.com", "raw.githubusercontent.com").replace("/blob/", "/")
    return url

def load_data(url):
    if url:
        return pd.read_csv(direct_link(url), encoding="utf-8-sig")
    if os.path.exists("data.csv"):
        return pd.read_csv("data.csv", encoding="utf-8-sig")
    try:
        from google.colab import files
        uploaded = files.upload()
        return pd.read_csv(next(iter(uploaded)), encoding="utf-8-sig")
    except ImportError:
        raise FileNotFoundError("Put data.csv next to this notebook or fill in DATA_URL.")

df_raw = load_data(DATA_URL.strip())
df = df_raw.copy()          # we work on a copy and keep the original untouched
print(f"Loaded {len(df):,} rows × {df.shape[1]} columns")

In [ ]:
#@title 1.2 A first look: one row = one case | מבט ראשון: שורה אחת = תיק אחד
show_cols = ["caseCitation", "legalProcedure", "petitionerType1", "respondentType1",
             "dateOpen", "dateFinalDecision", "dispositioncourt", "winnercourt", "Nwords"]
df[show_cols].head(10)

### Codebook · מילון משתנים

<div dir="rtl" lang="he">

<p>מילון המשתנים (codebook) הוא המסמך החשוב ביותר בכל מחקר אמפירי: הוא מסביר מה כל עמודה מודדת ואיך היא קודדה.
הטבלה הבאה מציגה את המשתנים המרכזיים, את סוגם, <b>איך הם קודדו</b> ואת שיעור הערכים החסרים בכל אחד.</p>
<p>שימו לב לעמודה <code>coded by</code>: רוב המשתנים חולצו אוטומטית מאתר בית המשפט, אבל חמישה משתנים (אופן הסיום, התוצאה, הזוכה ושני משתני הנושא) <b>קודדו ידנית בידי סטודנטים למשפטים</b>. בבדיקה חוזרת של כ-20% מהתיקים נמצאה התאמה של מעל 90%. במילים אחרות: גם המשתנים ה"חשובים" ביותר כוללים טעויות מדידה.</p>

</div>

The codebook is the most important document in any empirical project: it says what each column measures and how it was coded.
The table below lists the key variables, their type, **how they were coded** and how much of each is missing.
Most variables were extracted automatically from the court's website, but five (disposition, outcome, winner and the two subject-matter variables)
were **hand-coded by law students**; a second coder re-checked about 20% of cases and found over 90% agreement. Even the most important variables contain measurement error.

In [ ]:
#@title 1.3 Codebook of the main variables | מילון המשתנים המרכזיים
codebook = [
 # column,              English,                                                   Hebrew,                                         type,            coded by
 ("caseId",             "Docket number (serial/year of filing)",                    "מספר תיק (מספר/שנת פתיחה)",                     "identifier",    "automated"),
 ("url",                "Link to the final decision (shared by merged cases)",      "קישור להחלטה הסופית (משותף לתיקים מאוחדים)",     "identifier",    "automated"),
 ("merged",             "How many cases were consolidated together (blank = not merged)", "כמה תיקים אוחדו יחד (ריק = לא אוחד)",       "count",         "automated"),
 ("legalProcedure",     "Procedure type as filed (HCJ, civil/criminal appeal…)",    "סוג ההליך כפי שהוגש (בג\"ץ, ע\"א, ע\"פ…)",      "nominal",       "automated"),
 ("LIssue",             "Broad legal department (all HCJ = Administrative)",        "מחלקה משפטית כללית (כל בג\"ץ = מנהלי)",          "nominal",       "automated"),
 ("issuecourt-1",       "Main subject matter of the case",                          "הנושא המשפטי העיקרי",                          "nominal",       "hand-coded"),
 ("petitionerType1",    "Type of first petitioner/appellant, inferred from the name","סוג העותר/המערער הראשון, לפי שמו",             "nominal",       "automated"),
 ("respondentType1",    "Type of first respondent, inferred from the name",         "סוג המשיב הראשון, לפי שמו",                    "nominal",       "automated"),
 ("NPetitioners",       "Number of petitioners/appellants",                         "מספר העותרים/המערערים",                        "count",         "automated"),
 ("NRespondents",       "Number of respondents",                                    "מספר המשיבים",                                "count",         "automated"),
 ("courtSourceInstance","Court whose decision is under review (blank in most HCJ)", "הערכאה שהחלטתה נבחנת (ריק ברוב תיקי בג\"ץ)",     "nominal",       "automated"),
 ("courtSourceDistrict","District of that court (blank in HCJ)",                    "מחוז הערכאה הקודמת (ריק בבג\"ץ)",                "nominal",       "automated"),
 ("dateOpen",           "Date filed in the Supreme Court",                          "תאריך הגשת התיק לעליון",                       "date",          "automated"),
 ("dateFinalDecision",  "Date of the final decision closing the case",              "תאריך ההחלטה הסופית שסגרה את התיק",            "date",          "automated"),
 ("numHearings",        "Number of oral hearings",                                  "מספר הדיונים בעל-פה",                          "count",         "automated"),
 ("dispositioncourt",   "How the case ended (merits, withdrawal, settlement…)",     "אופן סיום התיק (לגופו, מחיקה, פשרה…)",         "nominal",       "hand-coded"),
 ("outcomecourt",       "Legal outcome (rejected, accepted, partly accepted…)",     "תוצאה משפטית (נדחה, התקבל, התקבל חלקית…)",     "nominal",       "hand-coded"),
 ("winnercourt",        "Who won (Both = partial win of unknown degree)",           "מי זכה (Both = זכייה חלקית בהיקף לא ידוע)",    "nominal",       "hand-coded"),
 ("Nwords",             "Words in the final decision, all opinions, rounded to 100","מילים בהחלטה הסופית, כל חוות הדעת, מעוגל ל-100", "ratio",         "automated"),
 ("agreedOutcome",      "Unanimous, or at least one dissent (Split)",               "פה אחד, או לפחות דעה חולקת אחת",               "binary",        "derived"),
 ("Njudges",            "Panel size (3–9; larger panels signal salient cases)",     "גודל ההרכב (3–9; הרכב מורחב = תיק חשוב)",      "count / ordinal","automated"),
 ("nMinority",          "Number of dissenting justices",                            "מספר שופטי המיעוט",                           "count",         "derived"),
 ("jOpinionWriter",     "Justice writing the lead opinion (blank if per curiam)",   "כותב/ת חוות הדעת הראשונה (ריק אם אין חוות דעת אישית)", "nominal",  "automated"),
 ("justice1",           "Head of the panel (justice2…justice9: other members)",     "אב/אם ההרכב (justice2–9: שאר השופטים)",        "nominal",       "automated"),
]
cb = pd.DataFrame(codebook, columns=["variable", "description", "תיאור", "type", "coded by"]).set_index("variable")
cb["pandas dtype"] = [str(df[c].dtype) for c in cb.index]
cb["% missing"] = [round(100 * df[c].isna().mean(), 1) for c in cb.index]
cb["# distinct values"] = [df[c].nunique() for c in cb.index]
cb

### 1.4 Types of variables · סוגי משתנים

<div dir="rtl" lang="he">

<p>הסוג של משתנה קובע אילו חישובים הגיוניים לגביו:</p>
<ul>
<li><b>שמי (nominal)</b> – קטגוריות ללא סדר: סוג העותר, סוג ההליך. מתארים בשכיחויות ובאחוזים. ממוצע של "סוג עותר" הוא חסר משמעות.</li>
<li><b>סודר (ordinal)</b> – קטגוריות עם סדר, אבל המרחקים ביניהן לא בהכרח שווים.</li>
<li><b>כמותי (count / ratio)</b> – מספרים אמיתיים: מספר מילים, מספר ימים, מספר דיונים. אפשר לחשב ממוצע, חציון וסטיית תקן.</li>
<li><b>תאריך</b> – נשמר כטקסט עד שממירים אותו; אחרי ההמרה אפשר לחשב ממנו משכי זמן.</li>
<li><b>מזהה</b> – מספר תיק או קישור. נראה כמו מספר, אבל ממוצע שלו לא אומר דבר.</li>
</ul>
<p>שימו לב לעמודה <code>pandas dtype</code> בטבלה: המחשב לא יודע מה המשמעות של המשתנה. התאריכים נקראו כטקסט (<code>object</code>), ו-<code>Njudges</code> נראה כמספר רציף אף שיש לו רק 4 ערכים (3, 5, 7, 9).</p>

</div>

The *type* of a variable determines which calculations make sense:
**nominal** (unordered categories — describe with counts and %), **ordinal** (ordered categories), **numeric / count**
(real quantities — mean, median, SD), **dates** (text until converted) and **identifiers** (look like numbers, but their mean is meaningless).

Look at the `pandas dtype` column: the computer does not know what a variable *means*. Dates were read as text, and `Njudges` looks numeric
although it takes only four values (3, 5, 7, 9).

In [ ]:
#@title 1.5 Convert dates and create new variables | המרת תאריכים ויצירת משתנים חדשים
for col in ["dateOpen", "dateFinalDecision", "dateFArgument", "dateLArgument"]:
    df[col] = pd.to_datetime(df[col], format="%m/%d/%y")
# Lower-court dates: a few have a mistyped year (e.g. "0014-06-12"); they become missing (see 2.4)
src = pd.to_datetime(df["dateCaseSource"], format="%Y-%m-%d", errors="coerce")
df["dateCaseSource"] = src.where(src >= "1990-01-01")

df["year_open"] = df["dateOpen"].dt.year                                   # year the case was filed
df["days"] = (df["dateFinalDecision"] - df["dateOpen"]).dt.days            # case duration in days
df["on_merits"] = np.where(df["dispositioncourt"] == "On the merits", "Yes", "No")

# Did the petitioner/appellant get anything? A coding choice: 'Both' counts as success, 'Unknown' is missing.
df["petitioner_success"] = df["winnercourt"].map({"Petitioner": 1, "Both": 1, "Respondent": 0})

df[["dateOpen", "dateFinalDecision", "days", "year_open", "winnercourt", "petitioner_success"]].head()

<div dir="rtl" lang="he">

<p><b>החלטת קידוד:</b> המשתנה <code>petitioner_success</code> מקבל 1 כשהעותר זכה או כשהתוצאה "Both" (זכייה חלקית), 0 כשהמשיב זכה, וערך חסר כשהזוכה לא ידוע.
לפי מילון המשתנים, "Both" פירושו שהעותר זכה <b>בחלק</b> מהסעד (בהיקף לא ידוע), כולל בפשרות; "Respondent" כולל גם מחיקות שבהן ברור שהעותר לא קיבל דבר; ו-"Unknown" הוא בעיקר מחיקות מסיבה לא ידועה.
ההחלטה לספור "Both" כהצלחה היא של החוקר, ואפשר לחלוק עליה. נסו להגדיר אותה אחרת (למשל רק "Petitioner" = 1) ובדקו איך משתנות התוצאות בהמשך.</p>

</div>

**A coding decision:** `petitioner_success` is 1 if the petitioner won or the outcome was "Both" (partial win), 0 if the respondent won,
and missing if the winner is unknown. Per the codebook, "Both" means the petitioner won *part* of the remedy (degree unknown), including in settlements;
"Respondent" includes withdrawals where the petitioner clearly got nothing; "Unknown" is mostly withdrawals for unknown reasons. Counting "Both" as success is the researcher's choice and it can be challenged — try a stricter definition
(only "Petitioner" = 1) and see how later results change.

In [ ]:
#@title 1.6 Frequency table of a categorical variable | טבלת שכיחויות
variable = "legalProcedure" #@param ["legalProcedure", "LIssue", "petitionerType1", "respondentType1", "dispositioncourt", "outcomecourt", "winnercourt", "agreedOutcome", "Njudges", "courtSourceInstance", "courtSourceDistrict", "jOpinionWriter", "year_open", "on_merits"] {allow-input: true}
include_missing = True #@param {type:"boolean"}
check_cols(variable)

counts = df[variable].value_counts(dropna=not include_missing)
table = pd.DataFrame({"n": counts, "%": 100 * counts / counts.sum()})
display(table)

top = table.head(15).iloc[::-1]
ax = top["%"].plot.barh(color=BLUE, figsize=(8, 0.35 * len(top) + 1))
ax.set_xlabel("% of cases"); ax.set_ylabel(""); ax.set_title(f"Distribution of {variable}")
ax.set_yticklabels([("(missing)" if str(i) == "nan" else str(i)) for i in top.index])
plt.show()

## 2. Missing data and errors · נתונים חסרים ושגיאות

<div dir="rtl" lang="he">

<p>ערך חסר הוא לא תמיד "מידע שאבד". כדאי להבחין בין שלושה סוגים:</p>
<ol>
<li><b>לא רלוונטי (חסר מבני):</b> אין "שופט רביעי" בהרכב של שלושה שופטים.</li>
<li><b>לא ידוע / לא נרשם:</b> המידע קיים בעולם, אבל לא נאסף.</li>
<li><b>שגיאה:</b> ערך שנרשם, אבל הוא בלתי אפשרי.</li>
</ol>
<p>השאלה החשובה: <b>האם ההיעדר אקראי?</b> אם ערכים חסרים בעיקר בסוג מסוים של תיקים, כל חישוב שמתעלם מהם מתאר רק את התיקים האחרים.</p>

</div>

A missing value is not always "lost information". Distinguish between **not applicable** (structural — there is no "fourth justice" on a three-judge panel),
**unknown / not recorded**, and **errors** (a value that was recorded but is impossible).
The key question: **is the missingness random?** If values are missing mainly for one kind of case, any calculation that ignores them describes only the other cases.

In [ ]:
#@title 2.1 How much is missing in each column? | כמה חסר בכל עמודה?
missing = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
missing = missing[missing > 0]
print(f"{len(missing)} of {df.shape[1]} columns have missing values")

ax = missing.iloc[::-1].plot.barh(color=BLUE, figsize=(8, 9))
ax.set_xlabel("% missing"); ax.set_title("Share of missing values per column")
plt.show()

In [ ]:
#@title 2.2 Why is it missing? Missingness by group | למה זה חסר? היעדרות לפי קבוצה
#@markdown Try: `justice4` by `Njudges` · `LIssueArea` by `legalProcedure` · `dateFArgument` by `numHearings` · `jOpinionWriter` by `dispositioncourt` · `petitionerType1` by `year_open`
missing_var = "justice4" #@param ["justice4", "LIssueArea", "courtSourceInstance", "dateFArgument", "jOpinionWriter", "petitionerType1", "dateCaseSource", "merged"] {allow-input: true}
by = "Njudges" #@param ["Njudges", "legalProcedure", "numHearings", "dispositioncourt", "LIssue", "year_open", "petitionerType1"] {allow-input: true}
check_cols(missing_var, by)

t = (df.assign(is_missing=df[missing_var].isna())
       .groupby(by, dropna=False)["is_missing"]
       .agg(n_cases="size", pct_missing=lambda s: 100 * s.mean()))
display(t.sort_values("n_cases", ascending=False))

<div dir="rtl" lang="he">

<p><b>שאלות לדיון:</b></p>
<ul>
<li><code>justice4</code> חסר בדיוק בתיקים שבהם ישבו שלושה שופטים. האם זה "נתון חסר"? האם צריך להשלים אותו?</li>
<li><code>LIssueArea</code> ו-<code>courtSourceInstance</code> חסרים כמעט רק בבג"ץ. למה? (רמז: מאיזו ערכאה מגיעה עתירה לבג"ץ?)</li>
<li><code>dateFArgument</code> (תאריך הדיון הראשון) חסר בדיוק כשמספר הדיונים הוא 0.</li>
<li><code>jOpinionWriter</code> חסר בעיקר בתיקים שלא הוכרעו לגופם. לפי מילון המשתנים, הוא ריק גם כשההחלטה ניתנה "בשם כל ההרכב", בלי חוות דעת אישית.</li>
<li><code>petitionerType1</code> חסר בכ-4% מהתיקים: הסוג נגזר <b>משם</b> הצד, והוא ריק כשהשם לא מאפשר סיווג או כשיש בו שגיאת הקלדה. האם זה נראה אקראי לאורך השנים? לפי סוג ההליך?</li>
</ul>

</div>

**Discussion:** `justice4` is missing exactly in three-judge panels — is that really "missing data"?
`LIssueArea` and `courtSourceInstance` are missing almost only in High Court of Justice petitions — why? (Hint: which court does an HCJ petition come from?)
`dateFArgument` is missing exactly when there were 0 hearings. `jOpinionWriter` is missing mostly in cases not decided on the merits, and also in per curiam decisions.
`petitionerType1` (about 4% missing) is inferred from the party's *name* and is blank when the name gives no clue or has a typo — does that look random across years? across procedures?

In [ ]:
#@title 2.3 What do statistics do with missing values? | מה קורה לערכים חסרים בחישוב?
s = df["petitionerType1"]
print("Rows in the data:                    ", len(s))
print("Non-missing values (.count()):       ", s.count())
print("value_counts() — silently drops NaN:  total =", s.value_counts().sum())
print("value_counts(dropna=False):           total =", s.value_counts(dropna=False).sum())
print()
print("petitioner_success mean (NaN ignored):", round(df["petitioner_success"].mean(), 3),
      " — based on", df["petitioner_success"].count(), "of", len(df), "cases")

<div dir="rtl" lang="he">

<p>רוב הפונקציות פשוט <b>מתעלמות</b> מערכים חסרים, בלי להודיע. זה נוח, אבל זה אומר שכל ממוצע מחושב על קבוצה קטנה יותר של תיקים, ולא תמיד על אותה קבוצה. תמיד דווחו על כמה תצפיות (n) מבוסס כל מספר.</p>

</div>

Most functions silently **ignore** missing values. Convenient — but every mean is then computed on a smaller (and possibly different) set of cases. Always report the *n* behind each number.

In [ ]:
#@title 2.4 Impossible values: decisions before filing? | ערכים בלתי אפשריים: החלטה לפני פתיחת התיק?
bad = df["days"] < 0
print(f"Cases with a negative duration: {bad.sum()}")
display(df.loc[bad, ["caseCitation", "legalProcedure", "dateOpen", "dateFinalDecision", "days"]])

bad_src = df["dateCaseSource"].isna() & df_raw["dateCaseSource"].notna()
print(f"Lower-court decision dates with an impossible year (set to missing in 1.5): {bad_src.sum()}")
display(df_raw.loc[bad_src, "dateCaseSource"].value_counts().head(8).to_frame("n"))

check = df["nMajority"] + df["nMinority"] != df["Njudges"]
print(f"Cases where majority + minority ≠ panel size: {check.sum()}")
display(df.loc[check, ["caseCitation", "Njudges", "nMajority", "nMinority", "agreedOutcome"]])

<div dir="rtl" lang="he">

<p>ב-50 תיקים תאריך ההחלטה בערכאה הקודמת נרשם עם שנה בלתי אפשרית, למשל <code>0014</code> במקום <code>2014</code>. אותן הפכנו לערכים חסרים כבר בשלב ההמרה (1.5).
במשכי הזמן השליליים, ברוב המקרים תאריך ההחלטה חל בפברואר–מרץ של אותה שנה, וכנראה שנת ההחלטה שגויה. <b>אנחנו לא "מתקנים" לפי ניחוש</b>: נהפוך את משך הזמן לערך חסר, נתעד את ההחלטה, ונוכל לבדוק בהמשך אם היא משנה משהו.</p>

</div>

50 lower-court decision dates have an impossible year (e.g. `0014` for `2014`); we already set them to missing when converting (1.5). Among the negative durations, most decisions fall in February–March of the same year — probably a wrong year. **We don't 'fix' by guessing**: we set the duration to missing, document the decision, and can later check whether it matters.

In [ ]:
#@title 2.5 Clean: set impossible durations to missing | ניקוי: משכים בלתי אפשריים יהפכו לחסרים
df["days_error"] = df["days"] < 0
df.loc[df["days_error"], "days"] = np.nan
print("Negative durations left:", (df["days"] < 0).sum(), "| missing durations now:", df["days"].isna().sum())

### 2.6 Unit of analysis: case or decision? · יחידת הניתוח: תיק או פסק דין?

<div dir="rtl" lang="he">

<p>כשכמה תיקים מאוחדים, כולם מקבלים <b>אותו פסק דין</b>, וכל אחד מהם מופיע כשורה נפרדת עם אותו מספר מילים. פסק הדין הארוך ביותר בנתונים מופיע 10 פעמים!
אם השאלה היא "כמה ארוך פסק דין טיפוסי?", יחידת הניתוח צריכה להיות פסק דין, ולא תיק.</p>

</div>

When several cases are merged they share **one decision**, and each appears as a separate row with the same word count. The longest decision in the data appears 10 times.
If the question is "how long is a typical decision?", the unit of analysis should be the decision, not the case.

In [ ]:
#@title 2.7 Same data, two units of analysis | אותם נתונים, שתי יחידות ניתוח
longest = df[df["Nwords"] == df["Nwords"].max()]
display(longest[["caseCitation", "merged", "merged_first", "Nwords"]])

decisions = df.drop_duplicates("url")
comparison = pd.DataFrame({
    "rows": [len(df), len(decisions)],
    "mean Nwords": [df["Nwords"].mean(), decisions["Nwords"].mean()],
    "median Nwords": [df["Nwords"].median(), decisions["Nwords"].median()],
}, index=["one row per case", "one row per decision"])
comparison

## 3. Describing numeric variables · תיאור משתנים כמותיים

<div dir="rtl" lang="he">

<ul>
<li><b>ממוצע</b> – סכום חלקי מספר התצפיות. רגיש מאוד לערכים קיצוניים.</li>
<li><b>חציון</b> – הערך האמצעי: חצי מהתיקים מעליו וחצי מתחתיו. עמיד לערכים קיצוניים.</li>
<li><b>סטיית תקן</b> – כמה, בממוצע, התצפיות רחוקות מהממוצע.</li>
<li><b>רבעונים</b> – 25% ו-75%. הטווח ביניהם מכיל את "חצי התיקים האמצעיים".</li>
</ul>
<p>כשההתפלגות <b>אסימטרית</b> (זנב ארוך ימינה, כמו אורך פסקי דין), הממוצע גבוה בהרבה מהחציון, ומי שמדווח רק את הממוצע עלול להטעות.</p>

</div>

**Mean** — sensitive to extreme values. **Median** — the middle value, robust to extremes. **Standard deviation** — typical distance from the mean.
**Quartiles** — the middle half of cases lies between the 25th and 75th percentiles.
When a distribution is **skewed** (a long right tail, like decision length) the mean is far above the median, and reporting only the mean can mislead.

In [ ]:
#@title 3.1 Summary statistics and histogram | סטטיסטיקה תיאורית והיסטוגרמה
variable = "Nwords" #@param ["Nwords", "days", "numHearings", "NPetitioners", "NRespondents", "Njudges", "merged"] {allow-input: true}
one_row_per_decision = False #@param {type:"boolean"}
cut_at_percentile = 100 #@param {type:"slider", min:90, max:100, step:1}
log_scale = False #@param {type:"boolean"}
check_cols(variable)

d = subset(df, one_row_per_decision=one_row_per_decision)
x = d[variable].dropna()
summary = x.describe(percentiles=[.1, .25, .5, .75, .9]).to_frame(variable).T
summary["skewness"] = x.skew()
display(summary)

shown = x[x <= x.quantile(cut_at_percentile / 100)]
fig, ax = plt.subplots()
ax.hist(shown, bins=60, color=BLUE, edgecolor="white", linewidth=0.5)
ax.axvline(x.mean(), color=ORANGE, lw=2, ls="--", label=f"mean = {x.mean():,.0f}")
ax.axvline(x.median(), color="black", lw=2, label=f"median = {x.median():,.0f}")
if log_scale:
    ax.set_yscale("log")
ax.set_xlabel(variable); ax.set_ylabel("number of rows")
ax.set_title(f"Distribution of {variable}" + (f" (up to the {cut_at_percentile}th percentile)" if cut_at_percentile < 100 else ""))
ax.legend(); plt.show()

<div dir="rtl" lang="he">

<p><b>נסו:</b> הזיזו את <code>cut_at_percentile</code> ל-95 וראו איך ההתפלגות של <code>Nwords</code> נראית בלי הזנב. שימו לב שכל הערכים של <code>Nwords</code> הם כפולות של 100: המדידה מעוגלת, וזה חלק ממה שאנחנו יודעים (ולא יודעים) על המשתנה.</p>

</div>

**Try:** move `cut_at_percentile` to 95 to see `Nwords` without its tail. Note that every value of `Nwords` is a multiple of 100 — the measurement is rounded, which is part of what we know (and don't know) about the variable.

In [ ]:
#@title 3.2 Compare groups | השוואה בין קבוצות
variable = "days" #@param ["days", "Nwords", "numHearings", "NPetitioners", "NRespondents"] {allow-input: true}
group = "legalProcedure" #@param ["legalProcedure", "LIssue", "petitionerType1", "respondentType1", "dispositioncourt", "agreedOutcome", "Njudges", "year_open", "on_merits", "courtSourceInstance"] {allow-input: true}
min_group_size = 30 #@param {type:"integer"}
check_cols(variable, group)

g = df.groupby(group)[variable]
table = g.describe()[["count", "mean", "std", "25%", "50%", "75%"]].rename(columns={"50%": "median"})
table = table[table["count"] >= min_group_size].sort_values("median")
display(table)
dropped = g.size()[g.size() < min_group_size]
if len(dropped):
    say(f"Not shown (fewer than {min_group_size} cases): {', '.join(map(str, dropped.index))}",
        f"קבוצות קטנות מדי שלא הוצגו: {', '.join(map(str, dropped.index))}")

order = table.index.tolist()
fig, ax = plt.subplots(figsize=(8, 0.5 * len(order) + 1.5))
sns.boxplot(data=df[df[group].isin(order)], y=group, x=variable, order=order,
            color=BLUE, showfliers=False, width=0.6, ax=ax)
ax.set_title(f"{variable} by {group} (box = middle 50%, line = median; outliers hidden)")
ax.set_ylabel("")
plt.show()

In [ ]:
#@title 3.3 Look at the extreme cases | הסתכלו על המקרים הקיצוניים
variable = "days" #@param ["days", "Nwords", "numHearings", "NPetitioners", "NRespondents"] {allow-input: true}
which_end = "largest" #@param ["largest", "smallest"]
n = 10 #@param {type:"integer"}
check_cols(variable)

d = df.nlargest(n, variable) if which_end == "largest" else df.nsmallest(n, variable)
d[["caseCitation", "legalProcedure", "dispositioncourt", variable]]

<div dir="rtl" lang="he">

<p>מספרים קיצוניים הם הזדמנות לבדוק את הנתונים: האם זה תיק אמיתי ויוצא דופן, או שגיאה? פתחו את הקישור לפסק הדין (עמודה <code>url</code>) של אחד מהם ובדקו.</p>

</div>

Extreme values are a chance to check the data: is this a real, unusual case — or an error? Open the decision link (`url`) of one of them and check.

## 4. Estimation and uncertainty · אמידה ואי-ודאות

<div dir="rtl" lang="he">

<p>יש לנו כמעט את כל התיקים של התקופה. אז למה צריך אי-ודאות? משתי סיבות:</p>
<ol>
<li>לרוב אנחנו רוצים ללמוד על <b>התהליך</b> (איך בית המשפט פועל) ולא רק על התיקים שכבר היו. התיקים בנתונים הם "מדגם" מתוך מה שהתהליך יכול לייצר.</li>
<li>במחקר אמיתי בדרך כלל קוראים <b>מדגם</b>: 200 פסקי דין, ולא 16,000. כאן נוכל לראות מה קורה כשדוגמים מתוך אוכלוסייה שאנחנו מכירים.</li>
</ol>

</div>

We have almost every case from the period — so why worry about uncertainty? (1) We usually want to learn about the **process** (how the court behaves),
not only about past cases. (2) Real research usually reads a **sample** — 200 decisions, not 16,000. Here we can see what happens when we sample from a population we know.

In [ ]:
#@title 4.1 Simulation: what if we had read only a sample? | סימולציה: מה אם היינו קוראים רק מדגם?
variable = "days" #@param ["days", "Nwords", "numHearings", "petitioner_success"] {allow-input: true}
sample_size = 100 #@param {type:"slider", min:10, max:1000, step:10}
n_samples = 1000 #@param {type:"integer"}
check_cols(variable)

population = df[variable].dropna()
rng = np.random.default_rng(1)
sample_means = np.array([rng.choice(population.to_numpy(), sample_size, replace=False).mean() for _ in range(n_samples)])

fig, ax = plt.subplots()
ax.hist(sample_means, bins=40, color=BLUE, edgecolor="white", linewidth=0.5)
ax.axvline(population.mean(), color="black", lw=2, label=f"true mean = {population.mean():,.2f}")
ax.set_xlabel(f"mean of {variable} in a sample of {sample_size}")
ax.set_ylabel("number of samples")
ax.set_title(f"{n_samples} researchers, each reading {sample_size} random cases")
ax.legend(); plt.show()

say(f"Spread of the sample means (SD): <b>{sample_means.std():,.2f}</b> · Formula for the standard error, σ/√n: <b>{population.std() / np.sqrt(sample_size):,.2f}</b>",
    "פיזור ממוצעי המדגמים כמעט זהה למה שנוסחת טעות התקן (σ/√n) מנבאת. נסו להגדיל את גודל המדגם: הפיזור קטן כמו 1/√n.")

In [ ]:
#@title 4.2 One sample, one confidence interval | מדגם אחד, רווח סמך אחד
variable = "days" #@param ["days", "Nwords", "numHearings", "petitioner_success"] {allow-input: true}
sample_size = 100 #@param {type:"slider", min:10, max:1000, step:10}
seed = 7 #@param {type:"integer"}
check_cols(variable)

population = df[variable].dropna()
sample = population.sample(sample_size, random_state=seed)
m, lo, hi, se, n = mean_ci(sample)
say(f"Sample mean = <b>{m:,.2f}</b>, 95% CI = [<b>{lo:,.2f}</b>, <b>{hi:,.2f}</b>] (n = {n}). True mean = {population.mean():,.2f}.",
    "רווח סמך של 95%: אם נחזור על הדגימה פעמים רבות, כ-95% מהרווחים שנחשב יכילו את הערך האמיתי. שנו את ה-seed כמה פעמים: בערך פעם אחת מכל 20 הרווח 'יחטיא'.")

In [ ]:
#@title 4.3 Mean with a confidence interval, by group | ממוצע עם רווח סמך, לפי קבוצה
variable = "days" #@param ["days", "Nwords", "numHearings", "petitioner_success"] {allow-input: true}
group = "LIssue" #@param ["LIssue", "legalProcedure", "petitionerType1", "respondentType1", "agreedOutcome", "Njudges", "year_open", "on_merits"] {allow-input: true}
merits_only = False #@param {type:"boolean"}
min_group_size = 30 #@param {type:"integer"}
check_cols(variable, group)

d = subset(df, merits_only=merits_only)
rows = []
for name, x in d.groupby(group)[variable]:
    if x.count() >= min_group_size:
        m, lo, hi, se, n = mean_ci(x)
        rows.append({group: name, "mean": m, "95% CI low": lo, "95% CI high": hi, "n": n})
res = pd.DataFrame(rows).set_index(group).sort_values("mean")
display(res)

fig, ax = plt.subplots(figsize=(8, 0.45 * len(res) + 1.5))
ax.errorbar(res["mean"], range(len(res)), xerr=[res["mean"] - res["95% CI low"], res["95% CI high"] - res["mean"]],
            fmt="o", color=BLUE, ecolor=BLUE, capsize=4, ms=8)
ax.set_yticks(range(len(res))); ax.set_yticklabels(res.index)
ax.set_xlabel(f"mean {variable} (95% CI)"); ax.set_title(f"Mean {variable} by {group}" + (" — decided on the merits" if merits_only else ""))
plt.show()

<div dir="rtl" lang="he">

<p><b>נסו:</b> בחרו <code>petitioner_success</code> לפי <code>petitionerType1</code>. ממוצע של משתנה 0/1 הוא <b>שיעור</b>: שיעור התיקים שבהם העותר השיג משהו.
שימו לב לרוחב רווחי הסמך: קבוצה גדולה (Individual) מקבלת רווח צר, וקבוצה קטנה מקבלת רווח רחב.</p>

</div>

**Try:** `petitioner_success` by `petitionerType1`. The mean of a 0/1 variable is a **proportion** — the share of cases where the petitioner got something. Notice how the interval width depends on group size.

## 5. Relationships between variables · קשרים בין משתנים

<div dir="rtl" lang="he">

<p>איך בודקים קשר תלוי בסוג המשתנים:</p>
<ul>
<li><b>שני משתנים שמיים</b> → טבלה דו-ממדית (crosstab) עם אחוזים</li>
<li><b>כמותי לפי קבוצה</b> → השוואת ממוצעים או חציונים (ראו פרק 3)</li>
<li><b>שני משתנים כמותיים</b> → תרשים פיזור ומקדם מתאם</li>
</ul>

</div>

**Two categorical variables** → a crosstab with percentages. **Numeric by group** → compare means/medians (section 3). **Two numeric variables** → scatterplot and correlation coefficient.

In [ ]:
#@title 5.1 Crosstab | טבלה דו-ממדית
rows = "petitionerType1" #@param ["petitionerType1", "respondentType1", "legalProcedure", "LIssue", "dispositioncourt", "agreedOutcome", "Njudges", "year_open"] {allow-input: true}
cols = "winnercourt" #@param ["winnercourt", "outcomecourt", "dispositioncourt", "on_merits", "agreedOutcome", "LIssue"] {allow-input: true}
percent_of = "row" #@param ["row", "column", "counts"]
merits_only = False #@param {type:"boolean"}
check_cols(rows, cols)

d = subset(df, merits_only=merits_only)
counts = pd.crosstab(d[rows], d[cols], margins=True, margins_name="Total")
display(counts)

if percent_of == "counts":
    shown, fmt, label = pd.crosstab(d[rows], d[cols]), "d", "number of cases"
else:
    shown = pd.crosstab(d[rows], d[cols], normalize="index" if percent_of == "row" else "columns") * 100
    fmt, label = ".0f", f"% of {percent_of}"
fig, ax = plt.subplots(figsize=(1.3 * shown.shape[1] + 3, 0.5 * shown.shape[0] + 1.5))
sns.heatmap(shown, annot=True, fmt=fmt, cmap="Blues", cbar_kws={"label": label}, linewidths=1, linecolor="white", ax=ax)
ax.set_title(f"{rows} × {cols}" + (" — merits only" if merits_only else "")); ax.set_xlabel(""); ax.set_ylabel("")
plt.show()

### 5.2 Which cases reach a decision? Selection effects · אילו תיקים מגיעים להכרעה? הטיית בחירה

<div dir="rtl" lang="he">

<p>רק כ-55% מהתיקים מוכרעים <b>לגופו של עניין</b>. השאר נמחקים, נסגרים בפשרה או מסתיימים אחרת. אם נחשב "שיעור הצלחה" רק בתיקים שהוכרעו לגופם, אנחנו מסתכלים על קבוצה <b>שנבחרה</b>:
תיקים שבהם הצדדים לא הצליחו להגיע להסכמה, או שהעותר לא חזר בו. זה הרעיון שמאחורי מודל Priest–Klein.</p>

</div>

Only about 55% of cases are decided **on the merits**; the rest are withdrawn, settled or end otherwise. A "success rate" computed only on merits decisions
describes a **selected** group — cases the parties could not settle and the petitioner did not abandon (the intuition behind the Priest–Klein model).

In [ ]:
#@title 5.3 Petitioner success: all cases vs. merits decisions only | הצלחת העותר: כל התיקים מול הכרעות לגופו בלבד
group = "legalProcedure" #@param ["legalProcedure", "petitionerType1", "respondentType1", "LIssue"] {allow-input: true}
min_group_size = 100 #@param {type:"integer"}
check_cols(group)

all_cases = df.groupby(group)["petitioner_success"].agg(["mean", "count"])
merits = subset(df, merits_only=True).groupby(group)["petitioner_success"].agg(["mean", "count"])
comp = pd.DataFrame({"all cases": all_cases["mean"] * 100, "merits only": merits["mean"] * 100,
                     "n (all)": all_cases["count"], "n (merits)": merits["count"]})
comp = comp[comp["n (merits)"] >= min_group_size].sort_values("all cases")
comp[["n (all)", "n (merits)"]] = comp[["n (all)", "n (merits)"]].astype(int)
display(comp.round(1))

ax = comp[["all cases", "merits only"]].plot.barh(color=[BLUE, ORANGE], width=0.75, figsize=(8, 0.6 * len(comp) + 1.5))
ax.set_xlabel("% of cases where the petitioner got something"); ax.set_ylabel("")
ax.set_title(f"Petitioner success by {group}"); ax.legend(loc="lower right")
plt.show()

<div dir="rtl" lang="he">

<p><b>לדיון:</b> בבג"ץ שיעור ההצלחה בהכרעות לגופו נמוך מאוד (כ-5%), אבל בכלל התיקים הוא גבוה פי שלושה בערך. איך זה ייתכן? מה קורה לעתירות שבהן המדינה משנה את עמדתה לפני ההכרעה?
מה זה אומר על מחקר שמסתמך רק על פסקי דין "מנומקים"?</p>

</div>

**Discussion:** in HCJ petitions the success rate on the merits is very low (~5%), yet about three times higher across all cases. How can that be? What happens to petitions where the state changes course before a ruling? What does this mean for research based only on reasoned judgments?

### 5.4 Correlation · מתאם

<div dir="rtl" lang="he">

<p><b>מקדם המתאם של פירסון (r)</b> מודד עד כמה שני משתנים כמותיים נעים יחד <b>בקו ישר</b>: 1 = קשר חיובי מושלם, 0 = אין קשר ליניארי, ‎-1 = קשר שלילי מושלם.
<b>מתאם ספירמן</b> משתמש בדירוגים במקום בערכים עצמם, ולכן הוא עמיד יותר לערכים קיצוניים ולהתפלגויות אסימטריות.</p>

</div>

**Pearson's r** measures how closely two numeric variables move together **along a straight line** (1 = perfect positive, 0 = no linear relation, −1 = perfect negative). **Spearman's ρ** uses ranks, so it is robust to outliers and skew.

In [ ]:
#@title 5.5 Scatterplot and correlation | תרשים פיזור ומתאם
x_var = "numHearings" #@param ["numHearings", "days", "Nwords", "NPetitioners", "NRespondents", "Njudges", "year_open"] {allow-input: true}
y_var = "days" #@param ["days", "Nwords", "numHearings", "NPetitioners", "NRespondents", "Njudges"] {allow-input: true}
jitter = True #@param {type:"boolean"}
log_y = False #@param {type:"boolean"}
check_cols(x_var, y_var)

d = df[[x_var, y_var]].dropna()
r, p_r = stats.pearsonr(d[x_var], d[y_var])
rho, p_rho = stats.spearmanr(d[x_var], d[y_var])
say(f"Pearson r = <b>{r:.3f}</b> · Spearman ρ = <b>{rho:.3f}</b> · n = {len(d):,}")

rng = np.random.default_rng(0)
xj = d[x_var] + (rng.uniform(-0.3, 0.3, len(d)) if jitter else 0)
fig, ax = plt.subplots()
ax.scatter(xj, d[y_var], s=8, alpha=0.15, color=BLUE, edgecolors="none")
slope, intercept = np.polyfit(d[x_var], d[y_var], 1)
xs = np.linspace(d[x_var].min(), d[x_var].max(), 50)
ax.plot(xs, intercept + slope * xs, color=ORANGE, lw=2, label="best-fitting straight line")
if log_y:
    ax.set_yscale("log")
ax.set_xlabel(x_var); ax.set_ylabel(y_var); ax.set_title(f"{y_var} vs {x_var}"); ax.legend()
plt.show()

In [ ]:
#@title 5.6 Correlation matrix | מטריצת מתאמים
method = "spearman" #@param ["pearson", "spearman"]
num_vars = ["Nwords", "days", "numHearings", "NPetitioners", "NRespondents", "Njudges", "year_open"]
corr = df[num_vars].corr(method=method)
fig, ax = plt.subplots(figsize=(7, 5.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            linewidths=1, linecolor="white", square=True, ax=ax)
ax.set_title(f"{method.capitalize()} correlations")
plt.show()

In [ ]:
#@title 5.7 Does the correlation hold within each type of proceeding? | האם המתאם נשמר בתוך כל סוג הליך?
x_var = "numHearings" #@param ["numHearings", "Nwords", "Njudges", "NPetitioners", "NRespondents"] {allow-input: true}
y_var = "days" #@param ["days", "Nwords", "numHearings"] {allow-input: true}
check_cols(x_var, y_var)

main = ["High Court of Justice", "Civil Appeal", "Criminal Appeal", "Administrative Appeal"]
rows = [{"group": "ALL CASES", "r": df[x_var].corr(df[y_var]), "n": df[[x_var, y_var]].dropna().shape[0]}]
for proc in main:
    d = df[df["legalProcedure"] == proc]
    rows.append({"group": proc, "r": d[x_var].corr(d[y_var]), "n": d[[x_var, y_var]].dropna().shape[0]})
pd.DataFrame(rows).set_index("group").round(3)

<div dir="rtl" lang="he">

<p><b>מתאם אינו סיבתיות.</b> יותר דיונים קשורים בתיקים ארוכים יותר, אבל האם הדיונים <b>גורמים</b> להתארכות? אולי תיקים מורכבים גם דורשים יותר דיונים וגם נמשכים זמן רב יותר (משתנה מתערב).
דוגמה נוספת: הרכבים גדולים כותבים פסקי דין ארוכים יותר. האם גודל ההרכב "גורם" לאורך? לפי מילון המשתנים, הרכב מורחב "מעיד בדרך כלל על תיק בולט משפטית או פוליטית", כלומר חשיבות התיק משפיעה גם על גודל ההרכב וגם על אורך פסק הדין.</p>

</div>

**Correlation is not causation.** More hearings go with longer cases — but do hearings *cause* delay, or do complex cases need both (a confounder)? Likewise, larger panels write longer opinions — does panel size cause length? The codebook notes that enlarged panels “generally indicate a salient legal or political case” — salience drives both panel size and length.

## 6. Hypothesis tests · מבחני השערות

<div dir="rtl" lang="he">

<p>ההיגיון של מבחן השערות דומה להליך פלילי:</p>
<ul>
<li><b>השערת האפס (H0)</b> היא "חזקת החפות": אין הבדל, אין קשר.</li>
<li>שואלים: <b>אם H0 נכונה</b>, כמה סביר לראות נתונים קיצוניים לפחות כמו אלה שראינו? התשובה היא <b>ערך p</b>.</li>
<li>אם p קטן מסף שנקבע מראש (בדרך כלל 0.05), דוחים את H0, "מעבר לספק סביר".</li>
</ul>
<p><b>מה ערך p אינו:</b> הוא לא ההסתברות ש-H0 נכונה, והוא לא מודד כמה ההבדל <b>חשוב</b>. עם 16,000 תיקים גם הבדל זניח יכול להיות "מובהק".</p>

</div>

The logic resembles a criminal trial: **H0** is the "presumption of innocence" (no difference, no relationship). We ask: *if H0 were true*, how likely is data at least as extreme as ours? That is the **p-value**.
If p is below a pre-set threshold (usually 0.05) we reject H0.
**What a p-value is not:** it is not the probability that H0 is true, and it does not measure how *important* a difference is. With 16,000 cases even a trivial difference can be "significant".

In [ ]:
#@title 6.1 Compare two groups (t-test) | השוואת שתי קבוצות (מבחן t)
variable = "Nwords" #@param ["Nwords", "days", "numHearings", "petitioner_success", "NPetitioners"] {allow-input: true}
group = "agreedOutcome" #@param ["agreedOutcome", "LIssue", "legalProcedure", "petitionerType1", "respondentType1", "on_merits", "Njudges"] {allow-input: true}
group_a = "split" #@param ["split", "unanimous", "Civil Law", "Criminal Law", "Administrative Law", "Government", "Individual", "Business", "Yes", "No"] {allow-input: true}
group_b = "unanimous" #@param ["unanimous", "split", "Civil Law", "Criminal Law", "Administrative Law", "Government", "Individual", "Business", "Yes", "No"] {allow-input: true}
one_row_per_decision = True #@param {type:"boolean"}
merits_only = False #@param {type:"boolean"}
check_cols(variable, group)

d = subset(df, merits_only=merits_only, one_row_per_decision=one_row_per_decision)
a = d.loc[d[group].astype(str) == group_a, variable].dropna()
b = d.loc[d[group].astype(str) == group_b, variable].dropna()
if len(a) < 2 or len(b) < 2:
    raise ValueError(f"Not enough cases. Values of {group}: {sorted(d[group].dropna().astype(str).unique())}")

summary = pd.DataFrame({"n": [len(a), len(b)], "mean": [a.mean(), b.mean()], "median": [a.median(), b.median()],
                        "SD": [a.std(), b.std()]}, index=[group_a, group_b])
display(summary)

t, p = stats.ttest_ind(a, b, equal_var=False)          # Welch's t-test: does not assume equal variances
u, p_mw = stats.mannwhitneyu(a, b)                     # rank-based check, robust to skew
say(f"H0: the mean of <b>{variable}</b> is the same for {group_a} and {group_b}.",
    f"השערת האפס: הממוצע של {variable} זהה בשתי הקבוצות.")
say(f"Difference in means = <b>{a.mean() - b.mean():,.2f}</b> · Welch t = {t:.2f} · Mann–Whitney p = {p_mw:.4f}")
p_sentence(p)

fig, ax = plt.subplots(figsize=(6, 3))
for i, (lab, x) in enumerate([(group_a, a), (group_b, b)]):
    m, lo, hi, se, n = mean_ci(x)
    ax.errorbar(m, i, xerr=[[m - lo], [hi - m]], fmt="o", color=[BLUE, ORANGE][i], capsize=5, ms=9)
ax.set_yticks([0, 1]); ax.set_yticklabels([group_a, group_b]); ax.set_ylim(-0.7, 1.7)
ax.set_xlabel(f"mean {variable} (95% CI)"); ax.set_title("Group means with 95% confidence intervals")
plt.show()

<div dir="rtl" lang="he">

<p><b>למה <code>one_row_per_decision</code>?</b> מבחני השערות מניחים שהתצפיות <b>בלתי תלויות</b>. עשרה תיקים מאוחדים עם אותו פסק דין הם לא עשר תצפיות בלתי תלויות על אורך פסקי דין. נסו לכבות את האפשרות וראו מה משתנה.</p>

</div>

**Why `one_row_per_decision`?** Tests assume **independent** observations. Ten merged cases sharing one decision are not ten independent observations of decision length. Switch it off and see what changes.

In [ ]:
#@title 6.2 Is there a relationship between two categorical variables? (chi-square) | קשר בין שני משתנים שמיים (חי-בריבוע)
rows = "petitionerType1" #@param ["petitionerType1", "respondentType1", "legalProcedure", "LIssue", "agreedOutcome", "Njudges"] {allow-input: true}
cols = "petitioner_success" #@param ["petitioner_success", "winnercourt", "on_merits", "agreedOutcome", "dispositioncourt"] {allow-input: true}
merits_only = True #@param {type:"boolean"}
min_group_size = 30 #@param {type:"integer"}
check_cols(rows, cols)

d = subset(df, merits_only=merits_only)
ct = pd.crosstab(d[rows], d[cols])
small = ct.index[ct.sum(axis=1) < min_group_size].tolist()
ct = ct.drop(index=small)
if small:
    say(f"Dropped small groups: {small}", f"קבוצות קטנות שהוסרו: {small}")

display((pd.crosstab(d[rows], d[cols], normalize="index") * 100).drop(index=small).round(1).assign(n=ct.sum(axis=1)))
chi2, p, dof, expected = stats.chi2_contingency(ct)
say(f"H0: <b>{rows}</b> and <b>{cols}</b> are independent.", f"השערת האפס: {rows} ו-{cols} בלתי תלויים.")
say(f"χ² = {chi2:,.1f}, df = {dof} · Cramér's V (strength, 0–1) = <b>{cramers_v(ct):.3f}</b>")
if (expected < 5).mean() > 0.2:
    say("⚠ More than 20% of cells have expected counts below 5 — the test may be unreliable.",
        "⚠ ביותר מ-20% מהתאים השכיחות הצפויה נמוכה מ-5, ולכן המבחן עלול להיות לא אמין.")
p_sentence(p)

In [ ]:
#@title 6.3 More than two groups (ANOVA / Kruskal–Wallis) | יותר משתי קבוצות
variable = "days" #@param ["days", "Nwords", "numHearings"] {allow-input: true}
group = "LIssue" #@param ["LIssue", "legalProcedure", "petitionerType1", "respondentType1", "Njudges", "year_open"] {allow-input: true}
min_group_size = 30 #@param {type:"integer"}
check_cols(variable, group)

groups = {k: v.dropna() for k, v in df.groupby(group)[variable] if v.count() >= min_group_size}
display(pd.DataFrame({k: {"n": len(v), "mean": v.mean(), "median": v.median()} for k, v in groups.items()}).T)
F, p_anova = stats.f_oneway(*groups.values())
H, p_kw = stats.kruskal(*groups.values())
say(f"H0: all {len(groups)} groups have the same mean {variable}. ANOVA F = {F:,.1f} · Kruskal–Wallis p = {p_kw:.4f}",
    "השערת האפס: הממוצע זהה בכל הקבוצות. דחייה אומרת שלפחות קבוצה אחת שונה, אבל לא אומרת איזו.")
p_sentence(p_anova)

### 6.4 Significance depends on sample size · מובהקות תלויה בגודל המדגם

<div dir="rtl" lang="he">

<p>נשווה את משך הטיפול בערעורים מנהליים ובערעורים פליליים. ההבדל האמיתי קבוע, ורק מספר התיקים שאנחנו "קוראים" משתנה. שימו לב: במדגמים קטנים ההבדל הנאמד "קופץ", ועם הגדלת המדגם הוא מתייצב סביב כ-80 יום. ערך p יורד ככל שהמדגם גדל, אף שההבדל האמיתי לא השתנה. (0.000 פירושו קטן מ-0.0005.)</p>

</div>

We compare case duration in administrative vs. criminal appeals. The real difference is fixed; only the number of cases we 'read' changes. In small samples the estimated difference jumps around; as n grows it settles near the true ~80 days. The p-value falls as n grows although the true difference never changed. (0.000 means below 0.0005.)

In [ ]:
#@title 6.4 Same difference, different sample sizes | אותו הבדל, מדגמים בגדלים שונים
a_all = df.loc[df["legalProcedure"] == "Administrative Appeal", "days"].dropna()
b_all = df.loc[df["legalProcedure"] == "Criminal Appeal", "days"].dropna()
rows = []
for n in [20, 50, 100, 250, 500, 1000, min(len(a_all), len(b_all))]:
    a, b = a_all.sample(n, random_state=3), b_all.sample(n, random_state=3)
    rows.append({"cases per group": n, "difference in mean days": a.mean() - b.mean(),
                 "p-value": stats.ttest_ind(a, b, equal_var=False).pvalue})
pd.DataFrame(rows).set_index("cases per group").round(4)

### 6.5 Fishing for significance · "דיג" של תוצאות מובהקות

<div dir="rtl" lang="he">

<p>נחלק את התיקים <b>באקראי</b> לשתי קבוצות, כמו הטלת מטבע, ונבדוק אם יש ביניהן הבדל במשך הטיפול בתיק. אין שום הבדל אמיתי. ובכל זאת, אם נריץ 20 מבחנים, בערך אחד מהם צפוי לצאת "מובהק".
זו הסיבה לנסח את ההשערה <b>לפני</b> שמסתכלים על הנתונים, ולדווח על כל המבחנים שהורצו.</p>

</div>

We split cases **at random** (a coin flip) into two groups and test for a difference in case duration. There is no real difference — yet out of 20 tests about one is expected to come out 'significant'. That is why you state the hypothesis **before** looking, and report every test you ran.

In [ ]:
#@title 6.5 Twenty meaningless tests | עשרים מבחנים חסרי משמעות
n_tests = 20 #@param {type:"slider", min:5, max:100, step:5}
seed = 6 #@param {type:"integer"}
rng = np.random.default_rng(seed)
x = df["days"].dropna().to_numpy()
pvals = []
for _ in range(n_tests):
    coin = rng.integers(0, 2, len(x)).astype(bool)
    pvals.append(stats.ttest_ind(x[coin], x[~coin], equal_var=False).pvalue)
pvals = np.array(pvals)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.scatter(range(1, n_tests + 1), pvals, color=np.where(pvals < 0.05, ORANGE, BLUE), s=50, zorder=3)
ax.axhline(0.05, color="black", ls="--", lw=1, label="p = 0.05")
ax.set_xlabel("test number"); ax.set_ylabel("p-value"); ax.set_ylim(0, 1)
ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
ax.set_title(f"{(pvals < 0.05).sum()} of {n_tests} random splits came out 'significant'")
ax.legend(); plt.show()

## 7. Your own question · השאלה שלכם

<div dir="rtl" lang="he">

<p>בזוגות, נסחו שאלה מחקרית שאפשר לענות עליה בעזרת הנתונים. לפני שמריצים משהו, מלאו:</p>
<ol>
<li><b>שאלה:</b> מה אתם רוצים לדעת?</li>
<li><b>השערה:</b> מה אתם מצפים למצוא, ולמה?</li>
<li><b>משתנים:</b> אילו עמודות מודדות את מה שמעניין אתכם? עד כמה המדידה טובה?</li>
<li><b>אילו תיקים:</b> כולם? רק הכרעות לגופו? שורה לכל פסק דין?</li>
<li><b>שיטה:</b> טבלה, השוואת ממוצעים, מתאם, איזה מבחן?</li>
<li><b>תוצאה:</b> מספר, רווח סמך או ערך p, ו-n.</li>
<li><b>מגבלות:</b> נתונים חסרים, הטיית בחירה, משתנים מתערבים, החלטות קידוד.</li>
</ol>
<p>רעיונות: האם המדינה מצליחה יותר כשהיא עותרת? האם תיקים נמשכים יותר בשנים מסוימות? האם הרכבים מורחבים חלוקים בדעותיהם לעיתים קרובות יותר? האם יש הבדל בין מחוזות?</p>

</div>

In pairs, write down **before** running anything: (1) **question**; (2) **hypothesis** and why; (3) **variables** — which columns measure what you care about, and how well;
(4) **which cases** — all, merits only, one row per decision?; (5) **method**; (6) **result** — a number with a CI or p-value, and n; (7) **limitations** — missing data, selection, confounders, coding choices.

Ideas: Does the state do better as petitioner? Did durations change over the years? Are expanded panels more often split? Do lower-court districts differ?

In [ ]:
#@title 7.1 Quick explorer — picks a suitable table, chart and test | סייר מהיר: בוחר טבלה, תרשים ומבחן מתאימים
outcome = "petitioner_success" #@param ["petitioner_success", "days", "Nwords", "numHearings", "winnercourt", "dispositioncourt", "outcomecourt", "agreedOutcome", "on_merits"] {allow-input: true}
explanatory = "petitionerType1" #@param ["petitionerType1", "respondentType1", "legalProcedure", "LIssue", "Njudges", "year_open", "courtSourceInstance", "courtSourceDistrict", "jOpinionWriter", "numHearings", "NPetitioners"] {allow-input: true}
filter_column = "(none)" #@param ["(none)", "legalProcedure", "LIssue", "petitionerType1", "respondentType1", "year_open", "courtSourceDistrict"] {allow-input: true}
filter_value = "" #@param {type:"string"}
merits_only = False #@param {type:"boolean"}
one_row_per_decision = False #@param {type:"boolean"}
min_group_size = 30 #@param {type:"integer"}
check_cols(outcome, explanatory)

d = subset(df, merits_only, one_row_per_decision, filter_column, filter_value)[[outcome, explanatory]].dropna()
say(f"Using <b>{len(d):,}</b> rows (after filters and dropping missing values).", f"מספר השורות בניתוח: {len(d):,}")

y_num = pd.api.types.is_numeric_dtype(d[outcome]) and d[outcome].nunique() > 2
x_num = is_continuous(d[explanatory])

if y_num and x_num:                                     # numeric × numeric → correlation
    r, p = stats.pearsonr(d[explanatory], d[outcome]); rho = stats.spearmanr(d[explanatory], d[outcome])[0]
    say(f"Pearson r = <b>{r:.3f}</b>, Spearman ρ = <b>{rho:.3f}</b>")
    fig, ax = plt.subplots(); ax.scatter(d[explanatory], d[outcome], s=8, alpha=0.2, color=BLUE)
    ax.set_xlabel(explanatory); ax.set_ylabel(outcome); plt.show()
    p_sentence(p)

elif not x_num and (y_num or set(d[outcome].unique()) <= {0, 1}):   # numeric (or 0/1) by group → compare means
    keep = d[explanatory].value_counts()[lambda s: s >= min_group_size].index
    d = d[d[explanatory].isin(keep)]
    rows = []
    for k, x in d.groupby(explanatory)[outcome]:
        m, lo, hi, se, n = mean_ci(x)
        rows.append({explanatory: k, "mean": m, "CI low": lo, "CI high": hi, "median": x.median(), "n": n})
    res = pd.DataFrame(rows).set_index(explanatory).sort_values("mean"); display(res)
    fig, ax = plt.subplots(figsize=(8, 0.45 * len(res) + 1.5))
    ax.errorbar(res["mean"], range(len(res)), xerr=[res["mean"] - res["CI low"], res["CI high"] - res["mean"]],
                fmt="o", color=BLUE, capsize=4, ms=8)
    ax.set_yticks(range(len(res))); ax.set_yticklabels(res.index); ax.set_xlabel(f"mean {outcome} (95% CI)"); plt.show()
    groups = [x for _, x in d.groupby(explanatory)[outcome]]
    if d[outcome].nunique() == 2 and len(groups) > 1:   # 0/1 outcome → compare proportions with chi-square
        ct = pd.crosstab(d[explanatory], d[outcome]); chi2, p, dof, _ = stats.chi2_contingency(ct)
        say(f"Test: chi-square (comparing proportions), χ² = {chi2:,.1f}, df = {dof}, Cramér's V = <b>{cramers_v(ct):.3f}</b>")
    elif len(groups) == 2:
        p = stats.ttest_ind(*groups, equal_var=False).pvalue; say("Test: Welch t-test (two groups)")
    elif len(groups) > 2:
        p = stats.f_oneway(*groups).pvalue; say(f"Test: one-way ANOVA ({len(groups)} groups)")
    else:
        p = None; say("Only one group left — nothing to compare.")
    if p is not None: p_sentence(p)

else:                                                   # categorical × categorical → crosstab + chi-square
    if x_num:
        say(f"{explanatory} is numeric; it is cut into 4 equal-sized groups (quartiles).")
        d[explanatory] = pd.qcut(d[explanatory], 4, duplicates="drop")
    ct = pd.crosstab(d[explanatory], d[outcome]); ct = ct[ct.sum(axis=1) >= min_group_size]
    display((ct.div(ct.sum(axis=1), axis=0) * 100).round(1).assign(n=ct.sum(axis=1)))
    chi2, p, dof, _ = stats.chi2_contingency(ct)
    say(f"Test: chi-square, χ² = {chi2:,.1f}, df = {dof}, Cramér's V = <b>{cramers_v(ct):.3f}</b>")
    p_sentence(p)

In [ ]:
# 7.2 Your own code | הקוד שלכם
# A few useful one-liners — uncomment one (remove the #) and run:
# df["courtSourceDistrict"].value_counts()
# df.groupby("year_open")["days"].median()
# df[df["legalProcedure"] == "High Court of Justice"]["petitioner_success"].mean()
# pd.crosstab(df["Njudges"], df["agreedOutcome"], normalize="index")
# df.query("petitionerType1 == 'Government' and on_merits == 'Yes'")["winnercourt"].value_counts()

### 7.3 Asking the AI assistant · שימוש בעוזר הבינה המלאכותית

<div dir="rtl" lang="he">

<p>ב-Colab אפשר לבקש מ-Gemini לכתוב קוד בשפה טבעית (הכפתור ✦ או "Generate"). זה כלי מצוין, בתנאי שבודקים את התוצאה. לפני שמאמינים לתשובה, שאלו:</p>
<ul>
<li><b>אילו שורות נכללו?</b> האם הוא סינן משהו? איך טיפל בערכים חסרים?</li>
<li><b>מה יחידת הניתוח?</b> תיק או פסק דין?</li>
<li><b>איך הוגדר המשתנה?</b> למשל, מה נחשב "זכייה"?</li>
<li><b>איזה מבחן ולמה?</b> האם ההנחות שלו מתקיימות?</li>
<li><b>האם המספר הגיוני?</b> השוו לטבלה פשוטה שהפקתם בעצמכם.</li>
</ul>
<p>דוגמה לבקשה טובה: <i>"Using df, compare the median of days between Civil Appeal and Criminal Appeal cases, report n for each group, and explain how missing values were handled."</i></p>

</div>

In Colab you can ask Gemini to write code in plain language. It is a great tool — if you check its work. Before trusting an answer, ask:
**Which rows were included** and how were missing values handled? **What is the unit of analysis?** **How was the variable defined** (what counts as a "win")?
**Which test, and why** — do its assumptions hold? **Is the number plausible** compared with a simple table you made yourself?

## Appendix: pandas cheat sheet · נספח: פקודות שימושיות

<div dir="rtl" lang="he">

<table dir="rtl">
<tr><th>מה עושים</th><th>פקודה</th></tr>
<tr><td>מספר שורות ועמודות</td><td dir="ltr"><code>df.shape</code></td></tr>
<tr><td>שמות העמודות</td><td dir="ltr"><code>df.columns</code></td></tr>
<tr><td>שכיחויות</td><td dir="ltr"><code>df["col"].value_counts(dropna=False)</code></td></tr>
<tr><td>סטטיסטיקה תיאורית</td><td dir="ltr"><code>df["col"].describe()</code></td></tr>
<tr><td>ממוצע / חציון / סטיית תקן</td><td dir="ltr"><code>.mean()  .median()  .std()</code></td></tr>
<tr><td>אחוזון</td><td dir="ltr"><code>.quantile(0.9)</code></td></tr>
<tr><td>ספירת ערכים חסרים</td><td dir="ltr"><code>df["col"].isna().sum()</code></td></tr>
<tr><td>סינון שורות</td><td dir="ltr"><code>df[df["col"] == "value"]</code></td></tr>
<tr><td>סטטיסטיקה לפי קבוצה</td><td dir="ltr"><code>df.groupby("g")["col"].mean()</code></td></tr>
<tr><td>טבלה דו-ממדית</td><td dir="ltr"><code>pd.crosstab(df["a"], df["b"], normalize="index")</code></td></tr>
<tr><td>מתאם</td><td dir="ltr"><code>df["a"].corr(df["b"], method="spearman")</code></td></tr>
</table>

</div>